# Bacillota and Bacteroidota in IBD versus non-IBD stool metagenomes

**Question.** In baseline stool metagenomes from the HMP2 / IBDMDB cohort (Lloyd-Price et al., *Nature* 2019; NCBI PRJNA398089), do the relative abundances of Bacillota (formerly Firmicutes) and Bacteroidota (formerly Bacteroidetes), and the ratio between them, differ between participants with IBD and non-IBD controls?

**Design.** One sample per participant (earliest visit), equal numbers per group drawn with a fixed seed by `scripts/select_samples.py`. IBD pools Crohn's disease and ulcerative colitis.

**Statistics.** Two-sided Mann–Whitney U test per phylum, Benjamini–Hochberg correction across the phyla tested, rank-biserial correlation as the effect size, and bootstrap 95% intervals for the difference in medians. Non-parametric tests are used because relative abundances are bounded and rarely normal.

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
import statsmodels
import matplotlib
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

PROJECT_DIR = Path(os.environ.get("PROJECT_DIR", "..")).resolve()
BRACKEN_DIR = PROJECT_DIR / "results" / "bracken"
SAMPLES_FILE = PROJECT_DIR / "config" / "samples.tsv"
OUT_DIR = PROJECT_DIR / "results" / "summary"
OUT_DIR.mkdir(parents=True, exist_ok=True)

MIN_MEAN_ABUNDANCE = 0.01   # only phyla averaging at least 1% are tested
N_BOOT = 5000
SEED = 2026

# Older and newer NCBI phylum names, mapped to the current names
PHYLUM_ALIASES = {"Firmicutes": "Bacillota", "Bacteroidetes": "Bacteroidota",
                  "Proteobacteria": "Pseudomonadota", "Actinobacteria": "Actinomycetota",
                  "Verrucomicrobia": "Verrucomicrobiota", "Fusobacteria": "Fusobacteriota"}

## 1. Load Bracken phylum tables

In [ ]:
samples = pd.read_csv(SAMPLES_FILE, sep="\t", index_col="run")

tables = {}
for run in samples.index:
    f = BRACKEN_DIR / f"{run}.bracken"
    assert f.exists(), f"Missing Bracken output for {run}"
    b = pd.read_csv(f, sep="\t")
    assert (b["taxonomy_lvl"] == "P").all(), f"{f.name} is not a phylum-level table"
    tables[run] = b.set_index("name")["fraction_total_reads"]

abund = pd.DataFrame(tables).T.fillna(0.0)
abund = abund.rename(columns=PHYLUM_ALIASES).T.groupby(level=0).sum().T  # merge any alias duplicates
abund = abund.div(abund.sum(axis=1), axis=0)                           # re-close to 1 after merging
print(f"{abund.shape[0]} samples x {abund.shape[1]} phyla")
samples.groupby(["group", "diagnosis"]).size().rename("n").to_frame()

## 2. Group summaries and tests

In [ ]:
ibd = samples.index[samples["group"] == "IBD"]
ctl = samples.index[samples["group"] == "nonIBD"]
rng = np.random.default_rng(SEED)

def boot_median_diff(a, b, n=N_BOOT):
    diffs = [np.median(rng.choice(a, len(a))) - np.median(rng.choice(b, len(b))) for _ in range(n)]
    return np.percentile(diffs, [2.5, 97.5])

eps = 1e-6
fb = np.log2((abund["Bacillota"] + eps) / (abund["Bacteroidota"] + eps)).rename("log2_Bacillota_Bacteroidota")
features = abund.loc[:, abund.mean() >= MIN_MEAN_ABUNDANCE].join(fb)

rows = []
for feat in features.columns:
    a, b = features.loc[ibd, feat].values, features.loc[ctl, feat].values
    u, p = mannwhitneyu(a, b, alternative="two-sided")
    lo, hi = boot_median_diff(a, b)
    rows.append({"feature": feat,
                 "mean_IBD": a.mean(), "mean_nonIBD": b.mean(),
                 "median_IBD": np.median(a), "median_nonIBD": np.median(b),
                 "median_diff": np.median(a) - np.median(b), "ci95_low": lo, "ci95_high": hi,
                 "U": u, "rank_biserial": 2 * u / (len(a) * len(b)) - 1, "p_value": p})

table = pd.DataFrame(rows).set_index("feature")
table["q_value"] = multipletests(table["p_value"], method="fdr_bh")[1]
table.to_csv(OUT_DIR / "group_comparison.csv")
abund.join(samples[["group", "diagnosis"]]).to_csv(OUT_DIR / "phylum_abundance_by_sample.csv")
table.round(4)

The rank-biserial correlation here runs from -1 to 1; with this sign convention a positive value means IBD samples tend to rank higher than controls. Report the bootstrap interval alongside the q value: with twelve people per group the interval is the more honest picture of how large the shift might be.

## 3. Figures

In [ ]:
show = ["Bacillota", "Bacteroidota", "log2_Bacillota_Bacteroidota"]
fig, axes = plt.subplots(1, 3, figsize=(11, 3.8))
for ax, feat in zip(axes, show):
    data = [features.loc[ctl, feat], features.loc[ibd, feat]]
    ax.boxplot(data, widths=0.5, showfliers=False)
    for i, d in enumerate(data, start=1):
        ax.scatter(np.full(len(d), i) + rng.uniform(-0.12, 0.12, len(d)), d, s=14,
                   c="#2c3e50" if i == 1 else "#c0392b", alpha=0.8, zorder=3)
    ax.set_xticks([1, 2], ["non-IBD", "IBD"])
    ax.set_title(f"{feat.replace('_', ' ')}\nq = {table.at[feat, 'q_value']:.2g}", fontsize=9)
axes[0].set_ylabel("relative abundance")
axes[2].set_ylabel("log2 ratio")
fig.tight_layout()
fig.savefig(OUT_DIR / "bacillota_bacteroidota.png", dpi=200)

In [ ]:
order = abund.mean().sort_values(ascending=False).index[:8]
stack = abund[order].assign(Other=1 - abund[order].sum(axis=1))
stack = stack.loc[list(ctl) + list(ibd)]
fig, ax = plt.subplots(figsize=(10, 3.8))
bottom = np.zeros(len(stack))
for col in stack.columns:
    ax.bar(range(len(stack)), stack[col], bottom=bottom, width=0.9, label=col)
    bottom += stack[col].values
ax.axvline(len(ctl) - 0.5, c="k", lw=1)
ax.set(xticks=[], ylabel="relative abundance", title="Phylum composition (left: non-IBD, right: IBD)")
ax.legend(fontsize=7, frameon=False, bbox_to_anchor=(1, 1))
fig.tight_layout()
fig.savefig(OUT_DIR / "phylum_composition.png", dpi=200)

## 4. Assumptions and limits

These are relative abundances, so a fall in one phylum forces others up even if absolute counts are unchanged; the analysis cannot say whether Bacteroidota grew or Bacillota shrank in absolute terms. Kraken2 classifies only what its database contains, and the 8 GB Standard build is a capped subset of RefSeq, so unclassified reads are excluded and less-studied taxa are under-counted. Bracken redistributes reads assuming a fixed read length (100 bp here). HMP2 participants differ in age, medication and disease activity, none of which is adjusted for in a two-group test, and a single time point per person cannot separate cause from consequence. Pooling Crohn's disease and ulcerative colitis gains power but may hide differences between them. With twelve participants per group, modest real differences can easily miss significance.

In [ ]:
print("Python", sys.version.split()[0])
for mod in (np, pd, scipy, statsmodels, matplotlib):
    print(f"{mod.__name__:<12}{mod.__version__}")